# 7 The stream toolkit

<div class="bp-banner">
  <div class="bp-series">Introduction to the Bash Shell</div>
  <div style="display:flex;align-items:baseline;gap:14px;flex-wrap:wrap;">
    <span class="bp-title">Part II — Pipelines and text extraction</span>
    <span class="bp-meta">Notebook&nbsp;7</span>
  </div>
  <div style="margin-top:10px;max-width:62ch;color:#46506b;">
    The small single-purpose filters (wc, cut, sort, uniq, tr) that reshape the
    lines you select into counts, columns, and ordered tables.
  </div>
  <div class="bp-rule" style="display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;">
    <span class="bp-meta">Raymond Amador</span>
    <span class="bp-meta">v1.0.0&nbsp;·&nbsp;CC&nbsp;BY&nbsp;4.0 (text) / MIT (code)</span>
  </div>
</div>

In [1]:
# Hidden setup: stand at the repo root, source the validation gate. data/ is
# read-only; the one exercise that saves output writes into a fresh scratch/.
ROOT="$PWD"; while [ ! -f "$ROOT/tools/check.sh" ] && [ "$ROOT" != "/" ]; do ROOT="$(dirname "$ROOT")"; done
source "$ROOT/tools/check.sh"
set +H
cd "$ROOT"

## What this notebook is about

`grep` handed you the matching lines. These five tools **reshape** them: turning a
pile of lines into a count, a single column, an ordered list, a tally. Each does
exactly one thing, reads from stdin and writes to stdout, and is built to sit in a
pipe. (A column of energies is just a column of numbers, as ever: **no physics
needed.**)

This is the notebook where the Unix philosophy finally pays off in full. Until now
the pipelines were short, because there was little to chain. With five reshaping
filters in hand, real multi-stage pipelines arrive, and the chapter ends on the
single most useful one in all of shell work.

## A. `wc` — how much is there?

The simplest question: how big? `wc` counts lines, words, and bytes.

```{command-card} wc
```

In [2]:
wc -l data/trajectories/lj38-optimization.xyz

2520 data/trajectories/lj38-optimization.xyz


Two and a half thousand lines — far too many to read, which is the whole reason we
filter. Give `wc` a file and no flag and it prints all three counts at once:

In [3]:
wc data/logs/gr2hno3-nvt.log

  721  3336 42963 data/logs/gr2hno3-nvt.log


## B. `cut` — pick a column

Lines are often columns of data. `cut` keeps the columns you want. Those CP2K log
lines are conveniently `TOKEN| …`, so the `|` is a clean delimiter: `-d'|'` sets it,
`-f1` keeps the first field: the section name on each line.

```{command-card} cut
```

In [4]:
grep '|' data/logs/gr2hno3-nvt.log | cut -d'|' -f1 | head -n 5

 DBCSR


 DBCSR


 DBCSR


 DBCSR


 DBCSR


`cut` can also slice by character position with `-c`, ignoring fields entirely:

In [5]:
cut -c1-9 data/logs/gr2hno3-nvt.log | head -n 5

 DBCSR| C


 DBCSR| M


 DBCSR| M


 DBCSR| M


 DBCSR| M


There is a wall here, though, and it matters because it points straight at the next
notebook. `cut`'s delimiter is a **single character**, and it counts *every* space
as a field separator. The energy value is padded away from its label by a *run* of
spaces, so those spaces become a dozen empty fields and the value is marooned out at
field 22. Reach for it at any smaller, predictable field and you land on the wrong
column — here, the unit label:

In [6]:
grep -m1 'Total FORCE_EVAL' data/logs/gr2hno3-nvt.log | cut -d' ' -f9

(a.u.):


That is the label `(a.u.):`, not the energy — `cut` reaching its limit. The moment
columns are separated by *variable* whitespace, you want `awk`, which is the first
half of Notebook 8.

## C. `sort` — put it in order

`sort` orders lines. Pull the energies out (the Notebook 6 way), and order them:

```{command-card} sort
```

In [7]:
grep 'Total FORCE_EVAL' data/logs/gr2hno3-nvt.log | grep -oE '\-[0-9]+\.[0-9]+' | sort -n

-145.725465772527684


-144.026841371125641


-143.350064790371448


-142.878629961526173


-142.438732194981611


-142.246533543175843


Note the `-n`. Leave it off and you meet the flagship trap of this notebook. By
default `sort` compares lines as *text*, character by character, so it puts the
energies in the wrong order, because lexically `-142…` comes before `-145…` (the
`2` beats the `5`), even though `-145` is the *smaller* number:

In [8]:
grep 'Total FORCE_EVAL' data/logs/gr2hno3-nvt.log | grep -oE '\-[0-9]+\.[0-9]+' | sort

-142.246533543175843


-142.438732194981611


-142.878629961526173


-143.350064790371448


-144.026841371125641


-145.725465772527684


The two orders are reversed. The textbook version of the same bug: lexically, `10`
sorts before `2` (because `1` beats `2`). Whenever a column is numbers, reach for
`-n`.

## D. `uniq` — collapse duplicates

`uniq` folds repeated lines into one, and with `-c` counts how many there were.
Counting how often each section appears in the log is a natural fit, but watch
what happens if we forget one thing:

```{command-card} uniq
```

In [9]:
grep '|' data/logs/gr2hno3-nvt.log | cut -d'|' -f1 | tr -d ' ' | uniq -c

     23 DBCSR


     10 CP2K


     19 GLOBAL


      9 MEMORY


      5 EWALD


...

      8 ROT


      9 THERMOSTAT


      1 ENERGY


      1 MD_ENERGIES


      5 ENERGY


Look closely: `ENERGY` is counted twice, as `1` and then `5`. That is the
number-one `uniq` confusion: **it only looks at *adjacent* lines.** A section that
appears in two different places in the log shows up as two separate runs, each
counted on its own. The fix is to `sort` first, so all the identical lines are
brought together:

In [10]:
grep '|' data/logs/gr2hno3-nvt.log | cut -d'|' -f1 | tr -d ' ' | sort | uniq -c

     10 CP2K


     23 DBCSR


      6 ENERGY


      5 EWALD


     19 GLOBAL


     14 MD


      1 MD_ENERGIES


      9 MEMORY


      8 ROT


      9 THERMOSTAT


Now each section is counted once, correctly. `sort | uniq` is one of the most
common two-command phrases in the shell: almost any time you reach for `uniq`,
`sort` comes just before it.

## E. `tr` — transform characters

The last filter works on *characters*, not lines. `tr` translates one set of
characters to another, deletes a set, or squeezes runs of a character to one.

```{command-card} tr
```

Upper-casing is the textbook example: translate the lowercase range to the
uppercase one:

In [11]:
echo "total energy" | tr 'a-z' 'A-Z'

TOTAL ENERGY


`-s` squeezes runs down to one, which tidies the ragged whitespace `cut` choked on
earlier (a workaround, not a replacement for `awk`):

In [12]:
grep -m1 'Total FORCE_EVAL' data/logs/gr2hno3-nvt.log | tr -s ' '

 ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.): -142.246533543175843


And `-d` deletes: the everyday use is stripping the stray carriage returns that
Windows leaves on line ends (`\r`):

In [13]:
printf 'a line\r\n' | cat -A

a line^M$


There is the intruder: `^M` before the `$`. Now delete it:

In [14]:
printf 'a line\r\n' | tr -d '\r' | cat -A

a line$


The `^M` is gone, leaving a clean `$` line end. Note the scope boundary, the twin
of `cut`'s: `tr` maps character *sets*. It cannot replace a *string* or a pattern;
that is `sed`, the other half of Notebook 8.

## Exercises

A pipeline-heavy set, because that is the point of these tools. All reads are
read-only and deterministic; the one exercise that saves a result writes into a
fresh `scratch/`.

### Warm-up 1 (worked) — Count

Count the lines of a trajectory and of a log with `wc -l`, and the words of the log
with `-w`.

In [15]:
cd "$ROOT"

In [16]:
# (solution hidden on the public site)


2520 data/trajectories/lj38-optimization.xyz


721 data/logs/gr2hno3-nvt.log


In [17]:
check '[ "$(wc -l < data/trajectories/lj38-optimization.xyz)" -eq 2520 ] && [ "$(wc -l < data/logs/gr2hno3-nvt.log)" -eq 721 ]' \
      "the trajectory is 2520 lines and the log is 721"

✓ the trajectory is 2520 lines and the log is 721


### Warm-up 2 (your turn) — Cut a column

The log's lines are `TOKEN| …`. Use `cut` with `-d` and `-f` to keep just the
section token on every `|` line, and count how many *distinct* sections there are
(pipe through `sort -u`).

In [18]:
cd "$ROOT"

In [19]:
# (solution hidden on the public site)


CP2K


DBCSR


ENERGY


EWALD


GLOBAL


MD


MD_ENERGIES


MEMORY


ROT


THERMOSTAT


In [20]:
check '[ "$(grep "|" data/logs/gr2hno3-nvt.log | cut -d"|" -f1 | tr -d " " | sort -u | wc -l)" -eq 10 ]' \
      "cut pulled the token column; there are 10 distinct sections"

✓ cut pulled the token column; there are 10 distinct sections


### Applied 1 (your turn) — Sort numerically

Extract the energy column (`grep -oE`, from Notebook 6) and order it with `sort
-n`. The first line is then the lowest (most negative) energy. Try it once without
`-n` to watch the lexical mangling.

In [21]:
cd "$ROOT"

In [22]:
# (solution hidden on the public site)


-145.725465772527684


In [23]:
check '[ "$(grep "Total FORCE_EVAL" data/logs/gr2hno3-nvt.log | grep -oE "\-[0-9]+\.[0-9]+" | sort -n | head -n 1)" = "-145.725465772527684" ]' \
      "sort -n found the lowest energy, -145.72…"

✓ sort -n found the lowest energy, -145.72…


### Applied 2 (your turn) — Dedup and count

Count how many lines belong to each log section: `sort` the token column, then
`uniq -c`. Confirm that the **DBCSR** section is the busiest. (Remember: `sort`
*before* `uniq`.)

In [24]:
cd "$ROOT"

In [25]:
# (solution hidden on the public site)


     10 CP2K


     23 DBCSR


      6 ENERGY


      5 EWALD


     19 GLOBAL


     14 MD


      1 MD_ENERGIES


      9 MEMORY


      8 ROT


      9 THERMOSTAT


In [26]:
check '[ "$(grep "|" data/logs/gr2hno3-nvt.log | cut -d"|" -f1 | tr -d " " | sort | uniq -c | grep DBCSR | grep -oE "[0-9]+")" -eq 23 ]' \
      "sort | uniq -c counted DBCSR's 23 lines"

✓ sort | uniq -c counted DBCSR's 23 lines


### Applied 3 (worked) — `tr` cleanup

Three character transforms: squeeze the ragged spaces of an energy line, strip a
carriage return, and lower-case a stream.

In [27]:
cd "$ROOT"

In [28]:
# (solution hidden on the public site)


 ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.): -142.246533543175843


value$


warning


In [29]:
squeezed="$(grep -m1 'Total FORCE_EVAL' data/logs/gr2hno3-nvt.log | tr -s ' ')"
check '! printf "%s" "$squeezed" | grep -q "  " && [ "$(printf "x\r" | tr -d "\r" | wc -c)" -eq 1 ] && [ "$(echo WARNING | tr "A-Z" "a-z")" = "warning" ]' \
      "spaces squeezed, carriage return stripped, and the line lower-cased"

✓ spaces squeezed, carriage return stripped, and the line lower-cased


### Composite — putting it together (the frequency idiom)

The payoff. Build the single most useful pipeline in shell text-work, the
**ranked-frequency idiom**, to rank the log's sections from busiest to quietest:

```
grep '|' … | cut -d'|' -f1 | tr -d ' ' | sort | uniq -c | sort -rn
```

It chains `grep` (Notebook 6) with four of this notebook's filters and the pipe
(Notebook 4): select the `|` lines, cut the token, tidy it, sort so duplicates are
adjacent, count them with `uniq -c`, and finally `sort -rn` to rank by count,
biggest first. Save the ranking to `scratch/`.

In [30]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [31]:
# (solution hidden on the public site)


     23 DBCSR


     19 GLOBAL


     14 MD


     10 CP2K


      9 THERMOSTAT


      9 MEMORY


      8 ROT


      6 ENERGY


      5 EWALD


      1 MD_ENERGIES


In [32]:
check '[ "$(head -n 1 scratch/ranking.txt | tr -s " " | sed "s/^ //")" = "23 DBCSR" ]' \
      "the ranking puts DBCSR (23 lines) at the top"

✓ the ranking puts DBCSR (23 lines) at the top


### Optional stretch (your turn) — A deeper pipeline

Go one stage further. Take the energy column, `sort -n`, and keep the **three
lowest** with `head -n 3`: the three most-bound configurations. Then, for the
boundary lesson: try to pull the second *numeric* column out of an energy line with
`cut` and watch the ragged whitespace defeat it. That failure is precisely the case
for `awk`, next notebook.

In [33]:
cd "$ROOT"

In [34]:
# (solution hidden on the public site)


-145.725465772527684


-144.026841371125641


-143.350064790371448


In [35]:
lowest3="$(grep 'Total FORCE_EVAL' data/logs/gr2hno3-nvt.log | grep -oE '\-[0-9]+\.[0-9]+' | sort -n | head -n 3)"
check '[ "$(printf "%s\n" "$lowest3" | wc -l)" -eq 3 ] && [ "$(printf "%s\n" "$lowest3" | head -n 1)" = "-145.725465772527684" ] && [ "$(printf "%s\n" "$lowest3" | tail -n 1)" = "-143.350064790371448" ]' \
      "the three lowest energies came out in numeric order, lowest (-145.72…) first"

✓ the three lowest energies came out in numeric order, lowest (-145.72…) first


## Outlook

You can now reshape streams into counts, columns, and ordered tables, and you have
hit two walls on purpose. `cut` cannot handle ragged whitespace, and `tr` cannot
replace strings. Next (Notebook 8): **`awk` and `sed`** (field-aware processing and
stream editing), which clear both walls at once and carry the regular-expression
thread from Notebook 6 right through to the end of Part II.

```{compendium-new}
```

<div class="bp-banner" style="margin-top:30px;">
  <div class="bp-series">Take this notebook with you</div>
  <div style="font-size:14.5px;line-height:1.55;max-width:66ch;">
    Open a <b>live terminal</b> from the &ldquo;Practice here&rdquo; box in any
    section to run everything yourself; nothing to install. The published
    notebooks ship <b>without worked solutions</b>; if you would like the
    reference solutions (to teach from or to check your own work), get in
    touch: <a href="mailto:hello@ramador.me">hello@ramador.me</a>.
  </div>
</div>